# 04 — Training (Phases 4 + 5)

**Inputs needed:** Cropped volumes from notebook 01, radiomics + VaRFS outputs from notebook 03, `configs/default.yaml`.
**Outputs produced:** `models/saved/best_swinvit_model.pth`, `swinvit_deep_extractor.pth`, `data/deep_features.csv`, `models/saved/fused_cross_attention.pth`, `results/training_history_*.json`.
**Runtime:** Highly hardware-dependent; budget 1–2 hours on a single mid-range GPU for ~50 patients.


Full training walkthrough using the same modules as `scripts/run_training.py`.

**Phase 4 — 3D SwinViT.** Trains on cropped liver volumes with a `WeightedRandomSampler`,
MONAI 3D augmentations, and Focal Loss. Saves
`models/saved/best_swinvit_model.pth`, `swinvit_deep_extractor.pth`, and writes deep
vectors to `data/deep_features.csv`.

**Phase 5 — Cross-Attention Fusion.** Loads VaRFS-stable radiomics + SwinViT deep
vectors and trains the cross-attention fusion + classification head, saving
`models/saved/fused_cross_attention.pth`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.utils.config import ensure_dirs, load_config, set_seed
from src.utils.logger import setup_logger

cfg = load_config(ROOT / "configs" / "default.yaml")
ensure_dirs(cfg)
set_seed(int(cfg.get("seed", 42)))
setup_logger("hcc", log_file=Path(cfg["paths"]["logs_dir"]) / "training.log")

## Quickest path — call the runner module directly

These helpers are exactly what `scripts/run_training.py` calls. Use them when you
want the same artifacts as the CLI but with a notebook in front of them.

In [ ]:
import importlib.util
spec = importlib.util.spec_from_file_location("run_training", ROOT / "scripts" / "run_training.py")
run_training = importlib.util.module_from_spec(spec)
spec.loader.exec_module(run_training)

In [ ]:
run_training._train_swinvit(cfg)

In [ ]:
run_training._train_fusion(cfg)

## Manual training (more control)

Use the cells below if you want to manually compose the SwinViT trainer (e.g.
for a custom subset, fewer epochs, or a different optimizer).

In [ ]:
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

from src.data.dataset import HCCDataset, get_3d_augmentation, get_weighted_sampler
from src.models.classifier import FocalLoss
from src.models.swin_vit import SwinViT3D
from src.models.trainer import Trainer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = get_3d_augmentation(cfg)
dataset = HCCDataset(
    processed_dir=cfg["paths"]["processed_dir"],
    labels_csv=cfg["paths"]["labels_csv"],
    transform=transform,
    target_size=tuple(cfg["swin_vit"]["img_size"]),
)
indices = np.arange(len(dataset))
train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=dataset.labels if len(set(dataset.labels)) > 1 else None,
    random_state=int(cfg.get("seed", 42)),
)
sub = HCCDataset.__new__(HCCDataset)
sub.samples = [dataset.samples[i] for i in train_idx]
sub.labels = dataset.labels[train_idx]
sampler = get_weighted_sampler(sub) if cfg["training"].get("weighted_sampler", True) else None
train_loader = DataLoader(
    Subset(dataset, train_idx.tolist()),
    batch_size=int(cfg["training"]["batch_size"]),
    sampler=sampler,
    shuffle=sampler is None,
)
val_loader = DataLoader(
    Subset(dataset, val_idx.tolist()),
    batch_size=int(cfg["training"]["batch_size"]),
    shuffle=False,
)
len(train_loader.dataset), len(val_loader.dataset)

In [ ]:
model = SwinViT3D(
    img_size=tuple(cfg["swin_vit"]["img_size"]),
    patch_size=tuple(cfg["swin_vit"]["patch_size"]),
    in_channels=int(cfg["swin_vit"]["in_channels"]),
    embed_dim=int(cfg["swin_vit"]["embed_dim"]),
    depths=tuple(cfg["swin_vit"]["depths"]),
    num_heads=tuple(cfg["swin_vit"]["num_heads"]),
    window_size=tuple(cfg["swin_vit"]["window_size"]),
    mlp_ratio=float(cfg["swin_vit"]["mlp_ratio"]),
    drop_path_rate=float(cfg["swin_vit"]["drop_path_rate"]),
    dropout=float(cfg["swin_vit"]["dropout"]),
    num_classes=int(cfg["model"]["num_classes"]),
    use_checkpoint=bool(cfg["swin_vit"]["use_checkpoint"]),
)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=float(cfg["training"]["learning_rate"]),
    weight_decay=float(cfg["training"]["weight_decay"]),
)
fl = cfg["training"]["focal_loss"]
loss_fn = FocalLoss(alpha=float(fl["alpha"]), gamma=float(fl["gamma"]), reduction="mean")
trainer = Trainer(model, optimizer, loss_fn, device, cfg)
summary = trainer.fit(
    train_loader,
    val_loader,
    checkpoint_path=Path(cfg["paths"]["model_save_dir"]) / "best_swinvit_model.pth",
    history_path=Path(cfg["paths"]["results_dir"]) / "training_history_swinvit.json",
)
summary